In [2]:
from __future__ import annotations

from enum import Enum

from debugpy.common.log import LEVELS

import identifier

In [3]:
import errno
import sys

import lmdb
import time
from pathlib import Path
from typing import Iterable, Iterator, Optional
import logging
import os
import re
from importlib import reload

# from scripts.regsetup import description

import record_pb2
from meta import Run, Dir, File

In [20]:
from db import Db
# reload(db)

db_name = 'test-db'
path = Path(db_name)

readonly = True
readonly = False

create=False
# create=True

mydb = Db.open(path, readonly=readonly, create=create)
env = mydb.env

In [ ]:
env, env.flags(), env.path(), env.info()

In [17]:
env.close()

In [14]:
with env.begin(
    write=False,
    # buffers= ?
) as txn:
    with txn.cursor() as cur:
        for key, value in cur.iternext():
            if key.startswith(b'r:'):
                print(key, value)

b'r:0016' b'\x08\x01" run from jupyter for development*\x03bla5\xb6\xf3\x9d?=\xb6\xf3\x9d?B\x0binitializedP\x08X\x18b\x0b\n\x04fld1\x12\x03blab\x16\n\x04fld2\x12\x0esan 64\r\nder 19'


In [ ]:
reload(db)

In [10]:
run_id = '0016'
run = Run(
    run_id,
    path='',
    description='run from jupyter for development',
    specification='bla',
    start_time=1.234,
    end_time=1.234,
    duration=1.234,
    num_dirs=8,
    num_files=24,
    extra={'fld1': 'bla', 'fld2': 'san 64\r\nder 19'},
    status='initialized')

from db import Db
key = Db.mk_run_key(run_id)
value = Db.mk_run_rec(run)
# value = msg.SerializeToString()

key, value


(b'r:0016',
 b'\x08\x01" run from jupyter for development*\x03bla5\xb6\xf3\x9d?=\xb6\xf3\x9d?B\x0binitializedP\x08X\x18b\x0b\n\x04fld1\x12\x03blab\x16\n\x04fld2\x12\x0esan 64\r\nder 19')

In [11]:
mydb

In [12]:
mydb.put_run(run)

Stored run 0016


b'r:0016'
b'\x08\x01" run from jupyter for development*\x03bla5\xb6\xf3\x9d?=\xb6\xf3\x9d?B\x0binitializedP\x08X\x18b\x0b\n\x04fld1\x12\x03blab\x16\n\x04fld2\x12\x0esan 64\r\nder 19'


In [22]:
rune = mydb.get_run('0016')
rune

Run(id='0016', path='', description='run from jupyter for development', specification='bla', start_time=1.2339999675750732, end_time=1.2339999675750732, duration=1.2339999675750732, extra={'fld2': 'san 64\r\nder 19', 'fld1': 'bla'}, status='initialized', num_dirs=8, num_files=24, error='')

In [23]:
rune.extra.items()

dict_items([('fld2', 'san 64\r\nder 19'), ('fld1', 'bla')])

In [ ]:
with env.begin(write=True) as txn:
    txn.put(key, value)

txn

In [24]:
with env.begin(write=False) as txn:
    c = txn.cursor()
    c.first()

In [25]:
txn
# txn.stat()